# PayGuard — Model 2: Random Forest

Supervised, **bagging** ensemble — many independent decision trees, each trained on a random subset of data/features, with predictions combined by majority vote.

Expectation: should meaningfully beat Logistic Regression's precision (fewer false alarms) while keeping recall reasonably high, since it can capture non-linear interactions between features that a linear model can't.

In [1]:
import joblib
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from evaluation import evaluate_model, print_comparison_table

bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]
class_weight_dict = bundle["class_weight_dict"]
X_train_smote, y_train_smote = bundle["X_train_smote"], bundle["y_train_smote"]
X_train_under, y_train_under = bundle["X_train_under"], bundle["y_train_under"]

print("Bundle loaded. Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

Bundle loaded. Train shape: (64000, 21) | Test shape: (16000, 21)


## Variant 1: Class-Weighted (trained on original, unresampled data)

In [2]:
rf_weighted = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight=class_weight_dict,
    random_state=42, n_jobs=-1,
)
rf_weighted.fit(X_train_scaled, y_train)

y_pred_weighted = rf_weighted.predict(X_test_scaled)
y_score_weighted = rf_weighted.predict_proba(X_test_scaled)[:, 1]

result_weighted = evaluate_model(y_test, y_pred_weighted, "Random Forest", "Class-Weighted", y_score_weighted)


  Random Forest (Class-Weighted)
Precision: 0.7619   Recall: 0.7111   F1: 0.7356

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15945                  10
Actual Fraud                  13                  32

ROC-AUC: 0.9699

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00     15955
       Fraud       0.76      0.71      0.74        45

    accuracy                           1.00     16000
   macro avg       0.88      0.86      0.87     16000
weighted avg       1.00      1.00      1.00     16000



## Variant 2: SMOTE

In [3]:
rf_smote = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = rf_smote.predict(X_test_scaled)
y_score_smote = rf_smote.predict_proba(X_test_scaled)[:, 1]

result_smote = evaluate_model(y_test, y_pred_smote, "Random Forest", "SMOTE", y_score_smote)


  Random Forest (SMOTE)
Precision: 0.1667   Recall: 0.7333   F1: 0.2716

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15790                 165
Actual Fraud                  12                  33

ROC-AUC: 0.9683

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.99      0.99     15955
       Fraud       0.17      0.73      0.27        45

    accuracy                           0.99     16000
   macro avg       0.58      0.86      0.63     16000
weighted avg       1.00      0.99      0.99     16000



## Variant 3: Random Undersampling

In [4]:
rf_under = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_under.fit(X_train_under, y_train_under)

y_pred_under = rf_under.predict(X_test_scaled)
y_score_under = rf_under.predict_proba(X_test_scaled)[:, 1]

result_under = evaluate_model(y_test, y_pred_under, "Random Forest", "Undersampling", y_score_under)


  Random Forest (Undersampling)
Precision: 0.0348   Recall: 0.8889   F1: 0.0669

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           14844                1111
Actual Fraud                   5                  40

ROC-AUC: 0.9676

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.93      0.96     15955
       Fraud       0.03      0.89      0.07        45

    accuracy                           0.93     16000
   macro avg       0.52      0.91      0.52     16000
weighted avg       1.00      0.93      0.96     16000



## Compare the three preprocessing variants for this model

In [5]:
rf_results = [result_weighted, result_smote, result_under]
rf_comparison_df = print_comparison_table(rf_results)


  MODEL COMPARISON (sorted by F1)
        model        variant  precision  recall     f1  roc_auc  true_positives  false_positives  true_negatives  false_negatives
Random Forest Class-Weighted     0.7619  0.7111 0.7356   0.9699              32               10           15945               13
Random Forest          SMOTE     0.1667  0.7333 0.2716   0.9683              33              165           15790               12
Random Forest  Undersampling     0.0348  0.8889 0.0669   0.9676              40             1111           14844                5


## Feature importance

Unlike Logistic Regression's signed coefficients, Random Forest gives unsigned "how much did this feature help split decisions correctly" importances — useful to compare against the Logistic Regression coefficients: do the same features show up as influential, even though the method of measuring influence is completely different?

In [6]:
importance_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "importance": rf_weighted.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
10,hour_of_day,0.157796
12,sender_balance_error,0.123695
14,balance_ratio,0.113896
0,amount,0.101437
11,is_off_hours,0.099338
8,amount_deviation_ratio,0.097296
2,sender_balance_before,0.078228
13,receiver_balance_delta,0.056613
3,sender_balance_after,0.053803
5,receiver_balance_after,0.030322


## Save the best variant's model + results

In [7]:
os.makedirs("../models", exist_ok=True)

# Select the variant with the highest F1 (our selection metric).
# Raw recall alone can reward models that generate too many false alarms.
best_variant = rf_comparison_df.loc[
    rf_comparison_df["f1"].idxmax()
]

print(
    f"Best variant for Random Forest: "
    f"{best_variant['variant']} "
    f"(F1={best_variant['f1']}, recall={best_variant['recall']})"
)

variant_to_model = {"Class-Weighted": rf_weighted, "SMOTE": rf_smote, "Undersampling": rf_under}
best_model = variant_to_model[best_variant["variant"]]

joblib.dump(best_model, "../models/random_forest.pkl")
rf_comparison_df.to_csv("../models/random_forest_results.csv", index=False)
print("Saved model to ../models/random_forest.pkl")
print("Saved results to ../models/random_forest_results.csv")

Best variant for Random Forest: Class-Weighted (F1=0.7356, recall=0.7111)
Saved model to ../models/random_forest.pkl
Saved results to ../models/random_forest_results.csv


## Next: `04c_model_xgboost.ipynb`